In [ ]:
#exhaustive search for CEDRUS+C parameters

import collections
import csv 

tsec = 256
maxsigs = 2**64

if tsec == 128:
    max_size_s = 6304
    max_sign_s = 2110892*2
    max_vrfy_s = 12766
    max_size_f = 14904
    max_sign_f = 111925*2
    max_vrfy_f = 5294
    MaxHash = 23
    zb_max = 8
if tsec == 192:
    max_size_s = 13776
    max_sign_s = 3739477*2
    max_vrfy_s = 19140
    max_size_f = 33016
    max_sign_f = 177541*2
    max_vrfy_f = 7924
    MaxHash = 23
    zb_max = 8
if tsec == 256:
    max_size_s = 26096
    max_sign_s = 3389656*2
    max_vrfy_s = 14905
    max_size_f = 46884
    max_sign_f = 361626*2
    max_vrfy_f = 8119
    MaxHash = 23
    zb_max = 8
CounterBytes = 4

class memoized(object):
  def __init__(self,func):
    self.func = func
    self.cache = {}
    self.__name__ = 'memoized:' + func.__name__
  def __call__(self,*args):
    if not isinstance(args,collections.abc.Hashable):
      return self.func(*args)
    if not args in self.cache:
      self.cache[args] = self.func(*args)
    return self.cache[args]

F=RealField(tsec+100)

@memoized
def qhitprob(leaves, qs, r):
    p = F(1) / leaves
    return binomial(qs, r) * (p**r) * ((1-p)**(qs-r))

@memoized
def forgeryprob(b, r, k):
    base_prob = 1 - (1 - F(1)/F(2**b))**r
    return base_prob**k

@memoized
def compute_c_sum(w_ary, l, s):
    dp = [[0] * (s + 1) for _ in range(l + 1)]
    dp[0][0] = 1
    for i in range(1, l + 1):
        for j in range(s + 1):
            w=w_ary[i-1]
            dp[i][j] = sum(dp[i - 1][j - k] for k in range(min(w - 1, j) + 1))
    return dp[l][s]

def compute_mincost(h,d):
    y=h%d
    if y == 0:
        h_ary = [int(h/d)]*d
    else:
        h1 = int(h/d)
        h2 = ceil(h/d)
        k = h-h1*d
        h_ary = [h1]*(d-k) + [h2]*k
    sum_h = 0
    for i in h_ary:
        sum_h += 2 ** i
    return sum_h

def compute_mincost_ary(h,d):
    y=h%d
    if y == 0:
        h_ary = [2**int(h/d)]*d
    else:
        h1 = int(h/d)
        h2 = ceil(h/d)
        k = h-h1*d
        h_ary = [2**h1]*(d-k) + [2**h2]*k
    return tuple(h_ary)

def wots_c():
    final_results = []
    two_pow_tsec = 2 ** tsec
    for zero_bit in range(zb_max + 1):
        diff = tsec - zero_bit
        start_l = ceil(diff / 8)
        end_l = diff // 2
        for l in range(start_l, end_l):
            print(zero_bit, l)
            w_ary = compute_mincost_ary(diff, l)
            sum_w = sum(w_ary)
            diff_sum_l = sum_w - l
            S_wn = diff_sum_l // 2
            vrfy_cost_wots = (diff_sum_l + 1) // 2
            sign_cost_wots = S_wn + l + two_pow_tsec / compute_c_sum(w_ary, l, S_wn)
            sign_cost_ht = sum_w + 1
            final_results.append([w_ary, zero_bit, l, sign_cost_wots, sign_cost_ht, vrfy_cost_wots])

    return final_results

def run_script():
    hashbytes = ceil(tsec/8)
    w_list = wots_c()
    with open("CEDRUS+C_256.csv","w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['h', 'd', 'b', 'k', 'w_ary', 'l', 'Size', 'sig_speed', 'vrfy_speed','z','fors_zero_bit','sec'])
        sigmalimit = F(2**(-tsec+MaxHash))
        donelimit = 1-sigmalimit/2**(20+MaxHash)
        s = log(maxsigs,2)
        for h in range(60, 69):
            leaves = 2**h
            for b in range(5,29):
                for k in range(1,64):
                    #print(h,b,k)
                    sigma = 0
                    r = 1
                    done = qhitprob(leaves, maxsigs, 0)
                    while done < donelimit:
                        t =qhitprob(leaves, maxsigs, r)
                        sigma += t*forgeryprob(b, r, k)
                        if sigma > sigmalimit:
                            break
                        done += t
                        r += 1
                    sigma += min(0, 1-done)
                    if sigma > sigmalimit:
                        continue
                    remove_tree_fors_bit = max(int(tsec+log(sigma,2))+1,0)
                    cost_fors_hash = 3*k*2**b-k+1
                    cost_fors_compress = 2**remove_tree_fors_bit
                    cost_fors_work = cost_fors_hash + cost_fors_compress
                    sigma = sigma*2**(-remove_tree_fors_bit)
                    if cost_fors_work<2**MaxHash and sigma<2**(-tsec):
                        for d in range(5,h):
                            print(h,b,k,d)
                            sign_cost_ht = compute_mincost(h, d) - d
                            for wots in w_list:
                                sign_speed = 1 + cost_fors_work + sign_cost_ht * wots[4] + d * wots[3]
                                if sign_speed <= max_sign_f:
                                    sig_size = ((b + 1) * k + h + wots[2] * d + 1) * hashbytes + CounterBytes * d
                                    if remove_tree_fors_bit > 0:
                                        sig_size += CounterBytes
                                    if sig_size <= max_size_f:
                                        vrfy_speed = 1 + k * (b + 1) + h + d * wots[5]
                                        if vrfy_speed <= max_vrfy_f:
                                            sec = -log(sigma, 2)
                                            writer.writerow([h,d,b,k,wots[0],wots[2],sig_size,float(sign_speed),vrfy_speed,wots[1],remove_tree_fors_bit,sec])
                                    continue
                                if sign_speed <= max_sign_s:
                                    sig_size = ((b + 1) * k + h + wots[2] * d + 1) * hashbytes + CounterBytes * d
                                    if remove_tree_fors_bit > 0:
                                        sig_size += CounterBytes
                                    if sig_size <= max_size_s:
                                        vrfy_speed = 1 + k * (b + 1) + h + d * wots[5]
                                        if vrfy_speed <= max_vrfy_s:
                                            sec = -log(sigma, 2)
                                            writer.writerow([h,d,b,k,wots[0],wots[2],sig_size,float(sign_speed),vrfy_speed,wots[1],remove_tree_fors_bit,sec])
run_script()
